# CURE-Rec advanced reviewer ablations
Run one guarded action at a time. Results are new evidence, not replacements for archived studies.


In [ ]:
from pathlib import Path
import sys, pandas as pd
CWD=Path.cwd().resolve()
CANDIDATES=[CWD, CWD/'paper-ideas'/'CURE-Rec'/'code', *CWD.parents]
ROOT=next((p for p in CANDIDATES if (p/'pyproject.toml').exists() and (p/'cure_rec').exists()), None)
if ROOT is None: raise RuntimeError('Open from the CURE-Rec code/notebooks directory or repository root.')
sys.path[:] = [str(ROOT), *[x for x in sys.path if x != str(ROOT)]]
from cure_rec.config import load_settings
from cure_rec.pipeline import run_experiment
from cure_rec.revision_advanced import select_objective, sampled_shapley
FULL_CONFIG=ROOT/'configs'/'curesim_full.yaml'


In [ ]:
RUN_OBJECTIVE_ABLATION=False
RUN_SAMPLED_SHAPLEY=False
SEED=300
assert not (RUN_OBJECTIVE_ABLATION and RUN_SAMPLED_SHAPLEY)


## Action 1 — maximin/mean and hard/penalty ablation


In [ ]:
if RUN_OBJECTIVE_ABLATION:
    cfg=load_settings(FULL_CONFIG); cfg.run.seed=SEED; cfg.run.output_root=ROOT/'runs'
    _,game,_=run_experiment(cfg)
    rows=[select_objective(game,cfg,objective=o,constraint_mode=c) for o in ('maximin','mean') for c in ('hard','penalty')]
    display(pd.DataFrame(rows))
else: print('Disabled.')


## Action 2 — exact versus sampled Shapley


In [ ]:
if RUN_SAMPLED_SHAPLEY:
    cfg=load_settings(FULL_CONFIG); cfg.run.seed=SEED; cfg.run.output_root=ROOT/'runs'
    _,game,_=run_experiment(cfg)
    exact=game.robust_shapley
    rows=[]
    for budget in (32,128,512,2048):
        est=sampled_shapley(game.robust_improvements,budget,seed=SEED)
        for player in exact: rows.append({'permutations':budget,'intervention':player,'exact':exact[player],'estimate':est[player],'absolute_error':abs(exact[player]-est[player])})
    display(pd.DataFrame(rows))
else: print('Disabled.')


CRN-off, scaling, user-level bootstrap, and a second dataset require additional simulator/evaluator implementations and should not be substituted with unvalidated shortcuts.
